# Week 8 Example Solution: Find Structure, Don't Fabricate It

This notebook demonstrates the full PCA + clustering workflow on the DASS-42 dataset.

**Phase 1:** PCA on 42 DASS items → scree plot → loadings → UMAP visualisation

**Phase 2:** k-Means clustering on Big Five + DASS profiles → silhouette evaluation → cluster profiling → stability checks

**Documentation:** AI-drafted methods paragraph with corrections

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import warnings

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

print('All imports loaded successfully!')

## 1. Load and Clean the Data

In [ ]:
# Load DASS-42 data (tab-separated)
data = pd.read_csv('../data/dass42_data.csv', sep='\t')
print(f'Raw dataset: {data.shape[0]:,} respondents, {data.shape[1]} variables')

# Filter unrealistic ages
data = data[(data['age'] >= 10) & (data['age'] <= 100)]
print(f'After age filter (10-100): {len(data):,}')

# Remove careless responders (VCL fake words)
data = data[(data['VCL6'] == 0) & (data['VCL9'] == 0) & (data['VCL12'] == 0)]
print(f'After VCL filter: {len(data):,}')

## 2. Score DASS Subscales and Big Five

In [ ]:
# Recode DASS items from 1-4 to 0-3
dass_cols = [f'Q{i}A' for i in range(1, 43)]
for col in dass_cols:
    data[col] = data[col] - 1

# Score three subscales (14 items each)
dep_items = [f'Q{i}A' for i in [3, 5, 10, 13, 16, 17, 21, 24, 26, 31, 34, 37, 38, 42]]
anx_items = [f'Q{i}A' for i in [2, 4, 7, 9, 15, 19, 20, 23, 25, 28, 30, 36, 40, 41]]
str_items = [f'Q{i}A' for i in [1, 6, 8, 11, 12, 14, 18, 22, 27, 29, 32, 33, 35, 39]]

data['DASS_Depression'] = data[dep_items].sum(axis=1)
data['DASS_Anxiety'] = data[anx_items].sum(axis=1)
data['DASS_Stress'] = data[str_items].sum(axis=1)

print('DASS subscales scored:')
for sub in ['DASS_Depression', 'DASS_Anxiety', 'DASS_Stress']:
    print(f'  {sub}: M={data[sub].mean():.1f}, SD={data[sub].std():.1f}')

In [ ]:
# Score Big Five from TIPI
for i in [2, 4, 6, 8, 10]:
    data[f'TIPI{i}R'] = 8 - data[f'TIPI{i}']

data['Extraversion'] = (data['TIPI1'] + data['TIPI6R']) / 2
data['Agreeableness'] = (data['TIPI2R'] + data['TIPI7']) / 2
data['Conscientiousness'] = (data['TIPI3'] + data['TIPI8R']) / 2
data['EmotionalStability'] = (data['TIPI4R'] + data['TIPI9']) / 2
data['Openness'] = (data['TIPI5'] + data['TIPI10R']) / 2

big5 = ['Extraversion', 'Agreeableness', 'Conscientiousness',
        'EmotionalStability', 'Openness']

print('Big Five scored:')
for trait in big5:
    print(f'  {trait}: M={data[trait].mean():.2f}, SD={data[trait].std():.2f}')

print(f'\nDataset ready: {len(data):,} respondents')

---

## Phase 1: Dimensionality Reduction

### 3. PCA on 42 DASS Items

In [ ]:
# Standardise the 42 DASS items
scaler = StandardScaler()
X_dass = scaler.fit_transform(data[dass_cols])

# Fit PCA
pca = PCA(random_state=42)
pca.fit(X_dass)

var_explained = pca.explained_variance_ratio_
cumulative_var = np.cumsum(var_explained)

print('PCA Results:')
print(f'  PC1: {var_explained[0]*100:.2f}%')
print(f'  PC2: {var_explained[1]*100:.2f}%')
print(f'  PC3: {var_explained[2]*100:.2f}%')
print(f'  Cumulative (3 PCs): {cumulative_var[2]*100:.2f}%')
print(f'  Components with eigenvalue > 1: {sum(pca.explained_variance_ > 1)}')

In [ ]:
# Scree plot
fig, ax1 = plt.subplots(figsize=(12, 6))

x = range(1, 16)
ax1.bar(x, var_explained[:15] * 100, color='#4A90D9', edgecolor='white', alpha=0.8,
        label='Individual variance')
ax1.set_xlabel('Principal Component', fontsize=13)
ax1.set_ylabel('Variance Explained (%)', fontsize=13, color='#4A90D9')
ax1.set_xticks(x)

ax2 = ax1.twinx()
ax2.plot(x, cumulative_var[:15] * 100, 'o-', color='#A71930', linewidth=2, markersize=6,
         label='Cumulative')
ax2.set_ylabel('Cumulative Variance (%)', fontsize=13, color='#A71930')
ax2.set_ylim(0, 100)

ax1.annotate('Elbow: 3 components\ncapture 55% of variance',
             xy=(3, var_explained[2] * 100), xytext=(7, var_explained[0] * 100 - 5),
             fontsize=11, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#2D2D2D', lw=1.5))

ax1.set_title('PCA Scree Plot — 42 DASS Items', fontsize=15, fontweight='bold', pad=15)
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right', fontsize=11)

plt.tight_layout()
fig.savefig('images/scree_plot.png', dpi=200, bbox_inches='tight')
plt.show()

### 4. Examine PCA Loadings

In [ ]:
# Loadings for first 3 components
loadings = pd.DataFrame(
    pca.components_[:3].T,
    columns=['PC1', 'PC2', 'PC3'],
    index=dass_cols
)

# Map items to subscales
subscale_map = {}
for item in dep_items: subscale_map[item] = 'Depression'
for item in anx_items: subscale_map[item] = 'Anxiety'
for item in str_items: subscale_map[item] = 'Stress'

# Top 5 loadings per component
for pc in ['PC1', 'PC2', 'PC3']:
    print(f'\n{pc} — top 5 loadings:')
    top = loadings[pc].abs().nlargest(5)
    for item, val in top.items():
        actual = loadings.loc[item, pc]
        sub = subscale_map[item]
        print(f'  {item} ({sub}): {actual:+.3f}')

In [ ]:
# Loadings heatmap
item_order = dep_items + anx_items + str_items
loadings_sorted = loadings.loc[item_order]

colour_map = {'Depression': '#A71930', 'Anxiety': '#4A90D9', 'Stress': '#5BA55B'}
row_colours = [colour_map[subscale_map[item]] for item in item_order]

fig, ax = plt.subplots(figsize=(8, 14))
sns.heatmap(loadings_sorted, annot=False, cmap='RdBu_r', center=0,
            xticklabels=True, yticklabels=True, ax=ax,
            cbar_kws={'shrink': 0.5, 'label': 'Loading'})

# Colour y-axis labels by subscale
for label in ax.get_yticklabels():
    item = label.get_text()
    if item in subscale_map:
        label.set_color(colour_map[subscale_map[item]])
        label.set_fontweight('bold')

ax.set_title('PCA Loadings — First 3 Components\n(items sorted by DASS subscale)',
             fontsize=14, fontweight='bold', pad=10)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#A71930', label='Depression'),
                   Patch(facecolor='#4A90D9', label='Anxiety'),
                   Patch(facecolor='#5BA55B', label='Stress')]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10)

plt.tight_layout()
fig.savefig('images/pca_loadings.png', dpi=200, bbox_inches='tight')
plt.show()

**Interpretation:**
- **PC1** (44.8%) = General distress — all items load positively. People who score high on one symptom tend to score high on everything.
- **PC2** (6.7%) = Depression vs Anxiety — Depression items load positively, Anxiety items load negatively.
- **PC3** (3.8%) = Stress vs Anxiety — Stress items load positively, Anxiety items load negatively.

The three DASS subscales do emerge as distinct components, but the dominant pattern is a single general distress factor.

### 5. UMAP Visualisation

In [ ]:
# Fit UMAP on standardised DASS items
print('Fitting UMAP (this may take a minute)...')
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embedding = reducer.fit_transform(X_dass)
print(f'UMAP embedding shape: {embedding.shape}')

fig, ax = plt.subplots(figsize=(12, 8))
scatter = ax.scatter(embedding[:, 0], embedding[:, 1],
                     c=data['DASS_Depression'].values,
                     cmap='RdYlGn_r', s=2, alpha=0.4)
cbar = plt.colorbar(scatter, ax=ax, shrink=0.8)
cbar.set_label('DASS Depression Score (0-42)', fontsize=12)

ax.set_xlabel('UMAP 1', fontsize=13)
ax.set_ylabel('UMAP 2', fontsize=13)
ax.set_title('UMAP Projection — Coloured by Depression Severity',
             fontsize=14, fontweight='bold')

ax.text(0.02, 0.02,
        'Warning: distances between clusters are NOT meaningful.\n'
        'Only local structure (nearby points) is trustworthy.',
        transform=ax.transAxes, fontsize=9, fontstyle='italic',
        color='#8C8C8C', verticalalignment='bottom',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

plt.tight_layout()
fig.savefig('images/umap_depression.png', dpi=200, bbox_inches='tight')
plt.show()

**Observation:** Depression severity appears as a smooth gradient in UMAP space, not as discrete clusters with sharp boundaries. This is consistent with the taxometric evidence that distress is dimensional.

---

## Phase 2: Clustering

### 6. k-Means: Evaluate Multiple k Values

In [ ]:
# Prepare clustering features: 5 Big Five + 3 DASS subscales
feature_cols = ['Extraversion', 'Agreeableness', 'Conscientiousness',
                'EmotionalStability', 'Openness',
                'DASS_Depression', 'DASS_Anxiety', 'DASS_Stress']

X = data[feature_cols].copy()
scaler_cluster = StandardScaler()
X_scaled = scaler_cluster.fit_transform(X)

# Try k = 2, 3, 4, 5, 6
k_values = [2, 3, 4, 5, 6]
inertias = []
silhouettes = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_scaled, labels)
    silhouettes.append(sil)
    print(f'k={k}: inertia={km.inertia_:,.0f}, silhouette={sil:.3f}')

In [ ]:
# Elbow and silhouette plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(k_values, inertias, 'o-', color='#4A90D9', linewidth=2, markersize=8)
ax1.set_xlabel('Number of Clusters (k)', fontsize=13)
ax1.set_ylabel('Inertia', fontsize=12)
ax1.set_title('Elbow Plot', fontsize=14, fontweight='bold')
ax1.set_xticks(k_values)

colours = ['#5BA55B' if s == max(silhouettes) else '#4A90D9' for s in silhouettes]
bars = ax2.bar(k_values, silhouettes, color=colours, edgecolor='white')
ax2.set_xlabel('Number of Clusters (k)', fontsize=13)
ax2.set_ylabel('Silhouette Score', fontsize=13)
ax2.set_title('Silhouette Score by k', fontsize=14, fontweight='bold')
ax2.set_xticks(k_values)

best_idx = silhouettes.index(max(silhouettes))
best_k = k_values[best_idx]
ax2.annotate(f'Best: k={best_k} (silhouette={max(silhouettes):.3f})',
             xy=(best_k, max(silhouettes)), xytext=(best_k + 1.2, max(silhouettes)),
             fontsize=11, fontweight='bold',
             arrowprops=dict(arrowstyle='->', color='#2D2D2D', lw=1.5))

plt.tight_layout()
fig.savefig('images/elbow_silhouette.png', dpi=200, bbox_inches='tight')
plt.show()

### 7. Profile the Best Cluster Solution

In [ ]:
# Fit best k
km_best = KMeans(n_clusters=best_k, random_state=42, n_init=10)
data['cluster'] = km_best.fit_predict(X_scaled)

# Profile: raw means per cluster
profile = data.groupby('cluster')[feature_cols].mean()
print('Cluster profiles (raw means):')
print(profile.round(2).to_string())
print()

for c in range(best_k):
    n = (data['cluster'] == c).sum()
    pct = n / len(data) * 100
    print(f'Cluster {c}: N={n:,} ({pct:.1f}%)')

In [ ]:
# Profile bar chart (standardised means)
profile_z = pd.DataFrame(
    [X_scaled[data['cluster'] == c].mean(axis=0) for c in range(best_k)],
    columns=feature_cols
)

cluster_labels = []
for c in range(best_k):
    dep_mean = profile.loc[c, 'DASS_Depression']
    cluster_labels.append(f'Cluster {c}: {"Low" if dep_mean < 15 else "High"} Distress')

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(feature_cols))
width = 0.35

colours = ['#5BA55B', '#A71930']
for c in range(best_k):
    offset = (c - (best_k - 1) / 2) * width
    ax.bar(x + offset, profile_z.iloc[c], width,
           label=cluster_labels[c], color=colours[c], edgecolor='white', alpha=0.85)

ax.set_xlabel('Feature', fontsize=13)
ax.set_ylabel('Standardised Mean (z-score)', fontsize=13)
ax.set_title(f'Cluster Profiles (k={best_k}) — Standardised Feature Means',
             fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([c.replace('DASS_', '') for c in feature_cols],
                   rotation=30, ha='right', fontsize=11)
ax.axhline(y=0, color='grey', linewidth=0.8, linestyle='--')
ax.legend(fontsize=11)

plt.tight_layout()
fig.savefig('images/cluster_profiles.png', dpi=200, bbox_inches='tight')
plt.show()

**Interpretation:** The two clusters split cleanly along a distress severity dimension:
- **Cluster 0 (Low distress):** Lower depression (11.8), anxiety (8.6), and stress (13.0). Higher Emotional Stability (4.18), Extraversion, and Conscientiousness.
- **Cluster 1 (High distress):** Much higher depression (29.7), anxiety (22.8), and stress (28.7). Lower Emotional Stability (2.32).

This is consistent with a *severity dimension* rather than distinct "types" of distress.

### 8. Stability Checks

In [ ]:
# Rerun k-means with different random seeds
ref_labels = data['cluster'].values
seeds = [0, 123, 456, 789, 2026]
stability_results = []

for seed in seeds:
    km_test = KMeans(n_clusters=best_k, random_state=seed, n_init=10)
    test_labels = km_test.fit_predict(X_scaled)
    
    # Check both label orderings (cluster labels can be permuted)
    match_same = np.sum(ref_labels == test_labels)
    match_flip = np.sum(ref_labels != test_labels)
    best_match = max(match_same, match_flip)
    agreement = best_match / len(ref_labels) * 100
    
    stability_results.append({'seed': seed, 'agreement': agreement})
    print(f'Seed {seed:4d}: {agreement:.2f}% agreement')

avg_agreement = np.mean([r['agreement'] for r in stability_results])
print(f'\nAverage agreement: {avg_agreement:.2f}%')

In [ ]:
# Stability bar chart
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar([f'Seed {r["seed"]}' for r in stability_results],
              [r['agreement'] for r in stability_results],
              color='#5BA55B', edgecolor='white', alpha=0.85)

ax.set_ylabel('Agreement with Reference (%)', fontsize=13)
ax.set_title(f'Cluster Stability — k={best_k} across Random Seeds',
             fontsize=14, fontweight='bold')
ax.set_ylim(95, 100.5)

for bar, result in zip(bars, stability_results):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
            f'{result["agreement"]:.1f}%', ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
fig.savefig('images/stability_check.png', dpi=200, bbox_inches='tight')
plt.show()

**Verdict:** The k=2 solution is extremely stable — 99.9% average agreement across random seeds. Fewer than 0.35% of participants change cluster in any run. This is what stability looks like when clusters are capturing a real pattern in the data.

---

## 9. Methods Paragraph

### AI Draft (with corrections in bold)

Responses from **34,576** participants were analysed using principal component analysis (PCA) and k-means clustering in Python (**scikit-learn 1.6**). After removing respondents with unrealistic age values (outside 10–100) **and those who endorsed one or more fake vocabulary check items (VCL6, VCL9, VCL12), indicating careless responding**, the 42 DASS items were standardised to z-scores and submitted to PCA. A scree plot and eigenvalue criterion indicated a three-component solution accounting for **55.36%** of total variance. The first component (44.78%) represented general distress with uniformly positive loadings, while Components 2 (6.74%) and 3 (3.85%) captured contrasts between Depression and Anxiety, and Stress and Anxiety, respectively. Participants were then clustered using k-means on eight standardised features (five Big Five personality scores and three DASS subscale totals). Silhouette analysis across k = 2–6 indicated an optimal two-cluster solution (silhouette = 0.240). The resulting clusters distinguished low-distress (N = 16,908; M Depression = 11.8) and high-distress (N = 17,668; M Depression = 29.7) profiles. Stability was assessed by rerunning k-means with five different random seeds, yielding 99.9% average agreement in cluster membership.

### What the AI got wrong

1. **Variance explained:** The AI initially said "56%" — the actual value is 55.36%
2. **Careless responder filtering:** The AI omitted this step entirely — we had to add it
3. **Software version:** The AI guessed an older scikit-learn version
4. **TIPI scale:** Not mentioned in the methods, but the AI would likely describe it as 1–7 (standard) rather than the 0–7 coding used in the OpenPsychometrics data

---

## 10. Save All Figures

All figures were saved to `images/` during the analysis above. Check the folder:
- `scree_plot.png`
- `pca_loadings.png`
- `umap_depression.png`
- `elbow_silhouette.png`
- `cluster_profiles.png`
- `stability_check.png`